# NER — Named Entity Recognition

Runs NER across the vault using the configured backend and compares results
against the ground truth in `data/ground_truth.yaml`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_ner

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
    and not Path(p).suffix == '.yaml'
)
print(f'{len(notes)} notes found')

## Run NER

In [ ]:
backend = config['run']['backend']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})
    gt_entities = [(e['entity'], e['type']) for e in gt_entry.get('entities', [])]

    row = {'file': name, 'lang': gt_entry.get('lang', '?'), 'ground_truth': gt_entities}

    try:
        row[backend] = [(e[0], e[1]) for e in run_ner(backend, post.content, config)]
    except Exception as e:
        row[backend] = f'ERROR: {e}'

    results.append(row)
    print(f'  {name}')

print(f'\nDone — {len(results)} notes processed')

## Summary comparison

Shows how many ground-truth entities each backend matched, plus how many extra it added.

In [ ]:
def ner_stats(gt_ents, pred):
    if isinstance(pred, str):
        return pred
    gt_set = set(gt_ents)
    pred_set = set(pred)
    matched = len(gt_set & pred_set)
    missed = len(gt_set - pred_set)
    extra = len(pred_set - gt_set)
    pct = int(100 * matched / len(gt_set)) if gt_set else 0
    return f'{matched}/{len(gt_set)} ({pct}%)  +{extra} extra'

rows = []
for r in results:
    rows.append({
        'file': r['file'],
        'lang': r['lang'],
        'gt_count': len(r['ground_truth']),
        backend: ner_stats(r['ground_truth'], r[backend]),
    })

pd.set_option('display.max_colwidth', 300)
pd.DataFrame(rows)

## Inspect a single note

Change `INSPECT` to any filename to see the full entity lists side by side.

In [ ]:
INSPECT = Path(notes[1]).name  # change to any note filename

row = next((r for r in results if r['file'] == INSPECT), None)
if row is None:
    print(f'{INSPECT} not found in results')
else:
    gt_set = set(row['ground_truth'])
    pred = set(row[backend]) if not isinstance(row[backend], str) else set()

    all_ents = sorted(gt_set | pred)
    rows = []
    for ent in all_ents:
        rows.append({
            'entity': ent[0],
            'type': ent[1],
            'ground_truth': '✓' if ent in gt_set else '',
            backend: '✓' if ent in pred else '',
        })

    print(f'=== {INSPECT} ===')
    pd.set_option('display.max_colwidth', 50)
    display(pd.DataFrame(rows))

## (Optional) Save results to frontmatter

Uncomment and run to write results to note files under `notemine.ner.<backend>`.
Run `python main.py reset` to undo.

In [ ]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     ner_data = notemine.get('ner', {})
#     if not isinstance(row.get(backend), str):
#         ner_data[backend] = [list(e) for e in row[backend]]
#     notemine['ner'] = ner_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')